In [0]:
%load_ext autoreload
%autoreload 2

# HUD USPS ZIP Code Crosswalk Ingestion

**Purpose:** One-time/infrequent ingestion of the HUD USPS ZIP-to-County crosswalk reference data.

**Run Frequency:** Quarterly (HUD publishes new crosswalk files quarterly)

**Source:** HUD USPS ZIP Code Crosswalk files
- Download page: https://www.huduser.gov/portal/pps_crosswalk.html
- URL pattern: https://www.huduser.gov/portal/datasets/usps/ZIP_COUNTY_YYYYMM.zip
- No API token needed for file download — files are public
- Format: ZIP file containing a CSV, or Excel (.xlsx) — notebook handles both
- HUD also has an API endpoint at /hudapi/public/usps (returns 400 without correct params)
  but the parameter docs are on the HUD website which is currently down. Once docs are
  available, the API may be a better option than file download.
- One row per ZIP x county pair, with allocation ratios

**Output File:**
* `/Volumes/bronze_dev/hud/hud_raw/reference_data/zip_county_crosswalk_YYYYMMDD_HHMMSS.csv`

**Note:** This raw CSV file will be picked up by the DLT bronze pipeline as a batch read (same pattern as states/counties reference data). See GOLD_PLAN.md §1c.

In [0]:
import requests
import zipfile
import io
import os
from datetime import datetime

In [0]:
# ============================================================================
# Configuration
# ============================================================================

# HUD USPS ZIP-county crosswalk download URL.
# Find the latest at: https://www.huduser.gov/portal/pps_crosswalk.html
# HUD publishes quarterly; URL pattern uses YYYYMM (e.g. 122024 = December 2024).
# The file may be .zip (containing CSV) or .xlsx — we try both.
CROSSWALK_URL = "https://www.huduser.gov/portal/datasets/usps/ZIP_COUNTY_122024.zip"

# Candidate URLs to try if the primary URL fails.
# Built from recent quarters, both .zip and .xlsx formats.
import itertools
_quarters = ["122024", "092024", "062024", "032024", "122023", "092023", "062023", "032023"]
_extensions = [".zip", ".xlsx"]
CANDIDATE_URLS = (
    [CROSSWALK_URL]  # try configured URL first
    + [f"https://www.huduser.gov/portal/datasets/usps/ZIP_COUNTY_{q}{ext}"
       for q, ext in itertools.product(_quarters, _extensions)]
)

# Volume path for raw reference data (same directory as states/counties)
VOLUME_PATH = "/Volumes/bronze_dev/hud/hud_raw/reference_data"

In [0]:
# ============================================================================
# Download the ZIP-county crosswalk file from HUD
# ============================================================================
# Try candidate URLs until we find one that returns actual content.
# HUD's server sometimes returns 202 with 0 bytes for non-existent files,
# so we verify content length > 0.

response = None
working_url = None

for url in CANDIDATE_URLS:
    try:
        print(f"Trying: {url}")
        r = requests.head(url, allow_redirects=True, timeout=20)
        content_length = int(r.headers.get("Content-Length", 0))
        print(f"  HEAD status={r.status_code}, content-length={content_length}")
        
        if r.status_code == 200 and content_length > 0:
            # This URL looks good — do the full GET
            working_url = url
            print(f"  -> Downloading...")
            r = requests.get(url, timeout=120)
            if r.status_code == 200 and len(r.content) > 0:
                response = r
                break
            else:
                print(f"  -> GET returned status={r.status_code}, bytes={len(r.content)}")
    except requests.exceptions.Timeout:
        print(f"  -> Timed out")
    except Exception as e:
        print(f"  -> Error: {e}")

if response is None:
    raise RuntimeError(
        f"Could not download crosswalk from any candidate URL. "
        f"Check https://www.huduser.gov/portal/pps_crosswalk.html "
        f"for the latest file and update CROSSWALK_URL manually."
    )

print(f"\nDownloaded {len(response.content):,} bytes from {working_url}")

# Detect file type from URL
is_xlsx = working_url.lower().endswith(".xlsx")
print(f"File format: {'Excel (.xlsx)' if is_xlsx else 'ZIP (.zip)'}")

In [0]:
# ============================================================================
# Extract CSV (from ZIP or XLSX) and save to volume
# ============================================================================

# Ensure the volume directory exists
os.makedirs(VOLUME_PATH, exist_ok=True)

if not is_xlsx:
    # ZIP file — extract CSV from inside
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        print(f"ZIP contents: {z.namelist()}")
        
        csv_files = [f for f in z.namelist() if f.lower().endswith('.csv')]
        
        if not csv_files:
            xlsx_in_zip = [f for f in z.namelist() if f.lower().endswith('.xlsx')]
            if xlsx_in_zip:
                raise RuntimeError(
                    f"Found Excel file ({xlsx_in_zip[0]}) inside ZIP but not CSV. "
                    f"Update this notebook to handle .xlsx inside .zip."
                )
            raise RuntimeError(f"No CSV file found in ZIP. Contents: {z.namelist()}")
        
        csv_filename = csv_files[0]
        csv_content = z.read(csv_filename)
        print(f"Extracted: {csv_filename} ({len(csv_content):,} bytes)")
else:
    # Excel file — convert to CSV using pandas
    import pandas as pd
    print("Converting .xlsx to CSV via pandas...")
    df_xlsx = pd.read_excel(io.BytesIO(response.content), engine='openpyxl')
    print(f"Excel sheet shape: {df_xlsx.shape}")
    print(f"Excel columns: {list(df_xlsx.columns)}")
    csv_content = df_xlsx.to_csv(index=False).encode('utf-8')
    print(f"Converted to CSV ({len(csv_content):,} bytes)")

# Generate timestamped filename and save
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_filename = f"zip_county_crosswalk_{timestamp}.csv"
output_path = f"{VOLUME_PATH}/{output_filename}"

with open(output_path, 'wb') as f:
    f.write(csv_content)

print(f"Saved ZIP-county crosswalk to: {output_path}")

In [0]:
# ============================================================================
# Verify: read back the saved CSV and show row count + sample
# ============================================================================

df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(output_path)
)

row_count = df.count()
print(f"Total rows: {row_count:,}")
print(f"Columns: {df.columns}")

# Show a few ZIPs that span multiple counties (TOT_RATIO < 1.0)
if "TOT_RATIO" in df.columns:
    multi_county = df.filter(df.TOT_RATIO < 1.0).count()
    print(f"ZIP x county rows where ZIP spans multiple counties: {multi_county:,}")

display(df.limit(10))